# Assignment Case A — HR Attrition & Workforce Analytics
## Working with PySpark DataFrames for Analytics
**Day 42 | Batch 13 | 03 Mei 2026**

---

| Field | Isi |
|-------|-----|
| **Nama** | *(isi nama kamu)* |
| **NIM** | *(isi NIM kamu)* |
| **Case** | Case A — HR Attrition & Workforce Analytics |
| **Source** | PostgreSQL AdventureWorks — HumanResources schema |
| **Target** | Hive: `adventureworks_curated.fact_hr_workforce` |

---

## ⚠️ Filter Penting

> **Kamu HANYA boleh menggunakan data dari Department: `Production` dan `Engineering`.**  
> Filter ini harus diterapkan sebelum transformasi.

## 🎯 Business Questions

1. **Department mana** (Production vs Engineering) yang memiliki rata-rata tenure karyawan lebih tinggi?
2. **Apakah ada korelasi** antara pay rate dan lama kerja karyawan?
3. **Berapa % karyawan** yang pernah pindah departemen minimal 1 kali?

## 📐 Pipeline yang Harus Dibangun

```
PostgreSQL (JDBC)  →  Spark DataFrame  →  Transform  →  Hive Curated Table
HumanResources.*      filter dept          TenureYears    fact_hr_workforce
                      join 5 tabel         IsMover
                                           DeptChangeCount
```

## 📋 Kolom Target: `fact_hr_workforce`

| Kolom | Tipe | Keterangan |
|-------|------|------------|
| `employee_id` | INT | Primary key |
| `full_name` | STRING | FirstName + LastName |
| `job_title` | STRING | Jabatan karyawan |
| `department_name` | STRING | Nama departemen (Production / Engineering) |
| `hire_date` | DATE | Tanggal mulai kerja |
| `tenure_years` | DECIMAL(5,2) | Lama kerja dalam tahun dari HireDate s/d sekarang |
| `latest_pay_rate` | DECIMAL(10,2) | Pay rate terbaru dari EmployeePayHistory |
| `is_mover` | BOOLEAN | TRUE jika pernah pindah departemen >= 1x |
| `dept_change_count` | INT | Total berapa kali pindah departemen |
| `load_timestamp` | TIMESTAMP | Waktu load |

## 🔗 Source Tables

| Tabel | Schema | Keterangan |
|-------|--------|------------|
| Employee | HumanResources | Data karyawan: JobTitle, HireDate |
| EmployeeDepartmentHistory | HumanResources | Riwayat perpindahan dept (EndDate NULL = aktif) |
| EmployeePayHistory | HumanResources | Riwayat pay rate — ambil terbaru |
| Department | HumanResources | Nama departemen — filter Production & Engineering |
| Person | Person | Nama lengkap karyawan |


## 0. Setup SparkSession

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from datetime import date

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('CaseA_HRAttrition') \
    .config('hive.metastore.uris', 'thrift://hive-metastore:9083') \
    .config('spark.sql.warehouse.dir', 'hdfs://namenode:9000/user/hive/warehouse') \
    .config('spark.jars', '/home/jovyan/work/jars/postgresql-42.7.3.jar') \
    .enableHiveSupport() \
    .getOrCreate()

# Referensi tanggal hari ini (untuk hitung TenureYears)
REFERENCE_DATE = date.today().isoformat()

# Filter department yang diperbolehkan
VALID_DEPARTMENTS = ['Production', 'Engineering']

print(f'Spark {spark.version} ready')
print(f'Reference date: {REFERENCE_DATE}')
print(f'Filter departments: {VALID_DEPARTMENTS}')
print('Spark UI → http://localhost:4040')

## 1. Load Data dari PostgreSQL via JDBC

Semua tabel source ada di PostgreSQL AdventureWorks (bukan Hive), jadi perlu baca via JDBC.

In [ ]:
# Konfigurasi JDBC ke PostgreSQL AdventureWorks
# Dari dalam Docker container: host.docker.internal
# Dari laptop langsung: localhost
JDBC_URL   = 'jdbc:postgresql://host.docker.internal:5433/Adventureworks'
JDBC_PROPS = {
    'user': 'postgres',
    'password': 'My_password1',
    'driver': 'org.postgresql.Driver'
}

def read_pg(schema, table):
    """Baca tabel dari PostgreSQL AdventureWorks"""
    return spark.read.jdbc(
        url=JDBC_URL,
        table=f'{schema}.{table}',
        properties=JDBC_PROPS
    )

# TODO: Load semua tabel yang dibutuhkan
df_employee   = ___  # HumanResources.Employee
df_edh        = ___  # HumanResources.EmployeeDepartmentHistory
df_pay        = ___  # HumanResources.EmployeePayHistory
df_dept       = ___  # HumanResources.Department
df_person     = ___  # Person.Person

print('Row counts:')
for name, df in [
    ('Employee', df_employee),
    ('EmployeeDepartmentHistory', df_edh),
    ('EmployeePayHistory', df_pay),
    ('Department', df_dept),
    ('Person', df_person),
]:
    print(f'  {name}: {df.count():,}')

## 2. Eksplorasi Data

In [ ]:
# TODO: Eksplorasi schema dan data
print('=== Schema Employee ===')
# TODO: df_employee.printSchema()

print('\n=== Schema EmployeeDepartmentHistory ===')
# TODO: df_edh.printSchema()

print('\n=== Semua Department yang tersedia ===')
# TODO: tampilkan semua nama department

print('\n=== Distribusi karyawan per department ===')
# TODO: join edh dengan dept, filter EndDate IS NULL, hitung count per dept name

## 3. Extract — Filter & Join

**⚠️ Filter Department Production & Engineering harus diterapkan di sini!**

Pipeline join yang diharapkan:
```
Employee
    JOIN EmployeeDepartmentHistory (EndDate IS NULL = departemen aktif)
    JOIN Department (filter Name IN ['Production','Engineering'])
    JOIN Person (ambil FirstName, LastName)
    JOIN LatestPayRate (subquery: ROW_NUMBER() OVER PARTITION BY EmployeeID ORDER BY RateChangeDate DESC)
    JOIN DeptHistoryCount (subquery: COUNT(*) GROUP BY EmployeeID)
```

In [ ]:
# ── Step 1: Filter department yang valid ─────────────────────────────────
df_dept_filtered = df_dept.filter(F.col('Name').isin(VALID_DEPARTMENTS))
print(f'Department yang lolos filter: {df_dept_filtered.count()}')
df_dept_filtered.show()

In [ ]:
# ── Step 2: Ambil departemen aktif per karyawan (EndDate IS NULL) ─────────
# TODO: Filter df_edh dimana EndDate IS NULL
df_edh_active = ___  # TODO
print(f'Active dept history rows: {df_edh_active.count():,}')

In [ ]:
# ── Step 3: Hitung pay rate terbaru per karyawan ──────────────────────────
# Gunakan Window function: ROW_NUMBER() OVER (PARTITION BY BusinessEntityID ORDER BY RateChangeDate DESC)
# Ambil hanya row dengan rn = 1

win_pay = Window.partitionBy('BusinessEntityID').orderBy(F.desc('RateChangeDate'))

df_latest_pay = df_pay \
    .withColumn('rn', F.row_number().over(win_pay)) \
    .filter(F.col('rn') == 1) \
    .select(
        F.col('BusinessEntityID').alias('pay_emp_id'),
        F.col('Rate').alias('latest_pay_rate'),
        F.col('RateChangeDate').alias('rate_change_date')
    )

print(f'Latest pay rate rows: {df_latest_pay.count():,}')
df_latest_pay.show(5)

In [ ]:
# ── Step 4: Hitung total dept history count per karyawan ─────────────────
# DeptHistoryCount = COUNT(*) GROUP BY BusinessEntityID dari tabel EmployeeDepartmentHistory
# (semua history, bukan hanya yang aktif)

# TODO: groupBy BusinessEntityID, hitung count semua record
df_hist_count = ___  # TODO

print(f'History count rows: {df_hist_count.count():,}')
df_hist_count.show(5)

In [ ]:
# ── Step 5: Join semua tabel ──────────────────────────────────────────────
# TODO: Gabungkan semua tabel
# Join order:
#   1. df_employee JOIN df_edh_active ON BusinessEntityID
#   2. JOIN df_dept_filtered ON DepartmentID
#   3. JOIN df_person ON BusinessEntityID
#   4. JOIN df_latest_pay ON BusinessEntityID
#   5. JOIN df_hist_count ON BusinessEntityID

df_enriched = ___  # TODO

print(f'Total rows setelah join: {df_enriched.count():,}')
df_enriched.printSchema()
df_enriched.show(5, truncate=False)

## 4. Transform — Hitung Kolom Turunan

### Aturan Kalkulasi:
- **tenure_years** = `ROUND((DATE_PART('year', AGE(NOW(), HireDate)) + DATE_PART('month', AGE(NOW(), HireDate))/12.0 + DATE_PART('day', AGE(NOW(), HireDate))/365.0), 2)`
- **is_mover** = `TRUE` jika `dept_history_count > 1`
- **dept_change_count** = `GREATEST(dept_history_count - 1, 0)` (posisi awal tidak dihitung)
- **full_name** = `CONCAT(FirstName, ' ', LastName)`

In [ ]:
# TODO: Hitung semua kolom turunan
# Petunjuk untuk tenure_years:
# F.datediff(F.current_date(), F.col('HireDate')) / 365.25
# Atau lebih presisi:
# (F.months_between(F.current_date(), F.col('HireDate')) / 12)

df_transformed = df_enriched \
    .withColumn('full_name',
        F.concat_ws(' ', F.col('FirstName'), F.col('LastName'))) \
    .withColumn('tenure_years',
        F.round(F.months_between(F.current_date(), F.col('HireDate')) / 12, 2)) \
    .withColumn('latest_pay_rate',
        F.round(F.col('latest_pay_rate'), 2)) \
    .withColumn('is_mover',
        ___) \
    .withColumn('dept_change_count',
        ___) \
    .select(
        F.col('BusinessEntityID').alias('employee_id'),
        F.col('full_name'),
        F.col('JobTitle').alias('job_title'),
        F.col('dept_name').alias('department_name'),
        F.col('HireDate').alias('hire_date'),
        F.col('tenure_years'),
        F.col('latest_pay_rate'),
        F.col('is_mover'),
        F.col('dept_change_count')
    )

print(f'Total rows setelah transform: {df_transformed.count():,}')
df_transformed.printSchema()
df_transformed.show(10, truncate=False)

In [ ]:
# Verifikasi filter department sudah benar
print('=== Department yang ada di hasil ===')
df_transformed.groupBy('department_name').count().show()

# Verifikasi IsMover
print('\n=== Distribusi IsMover ===')
df_transformed.groupBy('is_mover').count().show()

# Verifikasi tenure_years range
print('\n=== TenureYears Statistics ===')
df_transformed.select('tenure_years').describe().show()

## 5. Window Functions — Ranking per Department

In [ ]:
# TODO: Rank karyawan berdasarkan latest_pay_rate dalam tiap department
win_pay_rank = Window.partitionBy('department_name').orderBy(F.desc('latest_pay_rate'))

df_ranked = df_transformed \
    .withColumn('pay_rank_in_dept', F.rank().over(win_pay_rank)) \
    .withColumn('tenure_rank_in_dept',
        F.rank().over(Window.partitionBy('department_name').orderBy(F.desc('tenure_years'))))

print('=== Top 5 Karyawan per Department berdasarkan Pay Rate ===')
df_ranked.filter(F.col('pay_rank_in_dept') <= 5) \
         .orderBy('department_name', 'pay_rank_in_dept') \
         .select('department_name', 'full_name', 'job_title', 'latest_pay_rate',
                 'tenure_years', 'pay_rank_in_dept') \
         .show(20, truncate=False)

In [ ]:
# TODO: Pay bracket analysis — groupBy pay range untuk lihat pola tenure vs pay
df_pay_bracket = df_transformed \
    .withColumn('pay_bracket',
        F.when(F.col('latest_pay_rate') < 15, 'Low (<15)')
         .when(F.col('latest_pay_rate') < 25, 'Mid (15-25)')
         .when(F.col('latest_pay_rate') < 40, 'High (25-40)')
         .otherwise('Top (40+)')
    )

print('=== Rata-rata Tenure per Pay Bracket ===')
df_pay_bracket.groupBy('pay_bracket') \
    .agg(
        F.count('employee_id').alias('jumlah_karyawan'),
        F.round(F.avg('tenure_years'), 2).alias('avg_tenure_years'),
        F.round(F.avg('latest_pay_rate'), 2).alias('avg_pay_rate')
    ) \
    .orderBy('avg_pay_rate') \
    .show()

## 6. Load — Simpan ke Hive Curated

In [ ]:
spark.sql('CREATE DATABASE IF NOT EXISTS adventureworks_curated')

# Tambahkan load_timestamp
df_final = df_transformed.withColumn('load_timestamp', F.current_timestamp())

# TODO: Simpan ke adventureworks_curated.fact_hr_workforce
# mode: overwrite, saveAsTable

print('=== Verifikasi ===')
df_verify = spark.table('adventureworks_curated.fact_hr_workforce')
print(f'Total rows: {df_verify.count():,}')
df_verify.show(5, truncate=False)

## 7. Analytic Queries

**Wajib: minimal 3 query yang menjawab business question**

In [ ]:
# ── Query 1 (WAJIB): Rata-rata TenureYears per Department ─────────────────
# Jawab Business Question 1: department mana yang tenure rata-ratanya lebih tinggi?

print('=== Query 1: Rata-rata TenureYears per Department ===')
# TODO:

In [ ]:
# ── Query 2 (WAJIB): Top 5 Karyawan Pay Rate Tertinggi + Tenure ───────────
# Jawab Business Question 2: korelasi pay rate vs tenure

print('=== Query 2: Top 5 Karyawan Pay Rate Tertinggi ===')
# TODO:

In [ ]:
# ── Query 3 (WAJIB): Persentase IsMover = TRUE ────────────────────────────
# Jawab Business Question 3: berapa % karyawan yang pernah pindah dept?

print('=== Query 3: Persentase IsMover per Department ===')
# TODO:

In [ ]:
# ── Query 4 (BONUS): Korelasi Pay Rate vs Tenure — Group by Pay Bracket ───
# Apakah karyawan dengan pay rate lebih tinggi cenderung punya tenure lebih lama?

print('=== Query 4 (Bonus): Tenure per Pay Bracket ===')
# TODO:

In [ ]:
# ── Query 5 (BONUS): Distribusi JobTitle per Department ───────────────────

print('=== Query 5 (Bonus): Top 5 JobTitle per Department ===')
# TODO: groupBy department_name, job_title → count → rank per dept → filter rank <= 5
spark.sql("""
    -- TODO: tulis query SQL kamu di sini
    SELECT 1
""")

In [ ]:
# Cleanup
spark.stop()
print('Done! Cek tabel adventureworks_curated.fact_hr_workforce di DBeaver/HiveServer2.')